# YOLOv8 전이학습 — 협동로봇 파지 대상 물체 탐지

작성일: 2026-05-04  
대상 클래스: **필통, 장난감블록, 접시, 후추통** (총 4개)  
모델: `yolov8s.pt` (COCO pretrained) → 전이학습  
출력: `object_detection/resource/best.pt`

## 실행 순서

| 단계 | 섹션 | 설명 |
|------|------|------|
| 1 | 환경 설정 | GPU 확인, 경로 설정, ModelRegistry 초기화 |
| 2 | 데이터 준비 | Roboflow 다운 또는 직접 수집 구성 |
| 3 | YAML 생성 | 데이터셋 설정 파일 |
| 4 | 데이터 검증 | 이미지/라벨 수, 클래스 분포 확인 |
| 5 | 데이터 시각화 | 샘플 이미지 + bbox 오버레이 |
| 6 | 모델 학습 | 전이학습 실행 (버전 자동 부여) |
| 7 | 평가 & 버전 등록 | 지표 측정 → 레지스트리 기록 → best 자동 갱신 |
| 8 | 학습 곡선 | Loss / mAP 추이 그래프 |
| 9 | 클래스별 AP | 성능 취약 클래스 파악 |
| 10 | 예측 시각화 | GT vs 예측 bbox 비교 |
| 11 | 추론 속도 | FPS 벤치마크 |
| 12 | 버전 이력 | 전체 버전 성능 비교표 |

## 0. 환경 설정 & ModelRegistry

In [ ]:
import os
import json
import shutil
import time
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

import cv2
import numpy as np
import pandas as pd
import yaml
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

matplotlib.rcParams['font.family'] = ['NanumGothic', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 경로 ──────────────────────────────────────────────────────────────────
BASE_DIR    = Path(".").resolve()
DATA_DIR    = BASE_DIR / "data"
RUNS_DIR    = BASE_DIR / "runs"
VERS_DIR    = BASE_DIR / "versions"   # 버전별 PT 보관
YAML_PATH   = BASE_DIR / "cobot.yaml"
REGISTRY    = BASE_DIR / "model_registry.json"
OD_RESOURCE = BASE_DIR.parents[1] / "cobot2" / "object_detection" / "resource"

# ── 클래스 ────────────────────────────────────────────────────────────────
CLASS_NAMES = [
    "plate",       # 0: 접시
    "tissue",      # 1: 휴지
    "smartphone",  # 2: 스마트폰
    "remote",      # 3: 리모콘
    "glasses",     # 4: 안경
    "toy_block",   # 5: 장난감 블록 (Duplo)
    "shaker",      # 6: 조미료/향신료 통
]

COLORS = [
    (0.30, 0.90, 0.40),   # plate
    (0.95, 0.95, 0.95),   # tissue
    (0.20, 0.50, 0.95),   # smartphone
    (0.95, 0.65, 0.10),   # remote
    (0.75, 0.30, 0.85),   # glasses
    (0.95, 0.85, 0.20),   # toy_block (노랑)
    (0.65, 0.65, 0.70),   # shaker (실버)
]

# ── GPU ───────────────────────────────────────────────────────────────────
print("=" * 55)
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU  : {gpu}  VRAM {vram:.1f} GB")
else:
    print("  GPU  : 없음 — CPU 학습 (매우 느림)")
print(f"  BASE : {BASE_DIR}")
print(f"  OD   : {OD_RESOURCE}")
print("=" * 55)

### 성능 지표 기준

| 지표 | 역할 | best 선정 기준 |
|------|------|----------------|
| **fitness** | 주 선정 지표 = `0.1·mAP@50 + 0.9·mAP@50-95` | **높을수록 best** |
| mAP@50 | IoU 0.5 기준 평균 정밀도 (관대한 기준) | 참고용 |
| mAP@50-95 | IoU 0.5~0.95 평균 (엄격, COCO 공식) | 참고용 |
| Precision | 예측 중 정답 비율 | 참고용 |
| Recall | 실제 객체 검출 비율 | 참고용 |
| F1 | Precision·Recall 조화평균 | 참고용 |
| FPS | 실시간 처리 가능 여부 (≥10 필요) | 참고용 |
| center_err_px | bbox 중심 오차 평균(px) — 로봇 파지 직결 | 참고용 |

> **fitness 를 주 지표**로 사용하는 이유: YOLO 내부 fitness 함수와 동일 가중치(0.1·mAP50 + 0.9·mAP50-95).  
> mAP@50 만으로는 IoU 0.5 만 만족하면 통과 → bbox 중심 오차가 최대 ±50% 까지 허용된다.  
> 협동로봇 파지는 mm 단위 정확도가 필요하므로 mAP@50-95 가중치를 높게 두어 정밀한 bbox 예측 모델을 best 로 선정한다.

In [ ]:
class ModelRegistry:
    """
    버전 관리 + 성능 기록 + best.pt 자동 갱신.

    파일 레이아웃:
      model_registry.json     — 전체 버전 이력
      versions/v{N}_{ts}.pt   — 버전별 PT 보관본
      runs/best.pt            — 현재 최고 성능 모델

    best 기준 = fitness (= 0.1·mAP50 + 0.9·mAP50-95). YOLO 내부와 동일.
    """

    PRIMARY_METRIC = "fitness"

    def __init__(self, registry_path: Path, versions_dir: Path):
        self.path = registry_path
        self.versions_dir = versions_dir
        versions_dir.mkdir(parents=True, exist_ok=True)
        self._data = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            return json.loads(self.path.read_text())
        return {"versions": [], "best_version": None, "best_fitness": 0.0}

    def _save(self):
        self.path.write_text(json.dumps(self._data, indent=2, ensure_ascii=False))

    @staticmethod
    def fitness(metrics: dict) -> float:
        """YOLO 내부 fitness = 0.1·mAP50 + 0.9·mAP50-95"""
        return 0.1 * metrics.get("map50", 0.0) + 0.9 * metrics.get("map5095", 0.0)

    @staticmethod
    def env_snapshot() -> dict:
        """재현성용 환경 정보"""
        import sys, platform
        env = {
            "python": sys.version.split()[0],
            "platform": platform.platform(),
        }
        try:
            import torch
            env["torch"] = torch.__version__
            env["cuda"] = torch.version.cuda or "cpu"
        except Exception:
            pass
        try:
            import ultralytics
            env["ultralytics"] = ultralytics.__version__
        except Exception:
            pass
        return env

    @staticmethod
    def git_commit() -> str:
        """현재 git HEAD short hash (없으면 'none')"""
        import subprocess
        try:
            return subprocess.check_output(
                ["git", "rev-parse", "--short", "HEAD"],
                cwd=str(BASE_DIR), stderr=subprocess.DEVNULL
            ).decode().strip()
        except Exception:
            return "none"

    # ── 버전 번호 ─────────────────────────────────────────────────────────
    def next_version(self) -> str:
        n = len(self._data["versions"]) + 1
        return f"v{n}"

    def make_run_name(self, version: str) -> str:
        ts = datetime.now().strftime("%m%d_%H%M")
        return f"{version}_cobot_{ts}"

    # ── 데이터 fingerprint ────────────────────────────────────────────────
    @staticmethod
    def data_fingerprint(data_dir: Path) -> dict:
        """이미지 수 + 파일명 해시로 데이터 변경 여부 감지"""
        import hashlib
        fp = {}
        for split in ["train", "valid", "test"]:
            img_dir = data_dir / "images" / split
            files = sorted(p.name for p in img_dir.glob("*.[jp][pn]g")) if img_dir.exists() else []
            fp[f"{split}_count"] = len(files)
            fp[f"{split}_hash"]  = hashlib.md5("\n".join(files).encode()).hexdigest()[:8]
        return fp

    def detect_data_change(self, current_fp: dict) -> tuple[bool, str]:
        """직전 등록 버전과 fingerprint 비교 — (changed, message)"""
        if not self._data["versions"]:
            return False, "최초 학습"
        last_fp = self._data["versions"][-1].get("data_fingerprint", {})
        if not last_fp:
            return False, "이전 fingerprint 없음 (이전 버전이 구형)"
        changes = []
        for split in ["train", "valid", "test"]:
            oc = last_fp.get(f"{split}_count", 0)
            nc = current_fp.get(f"{split}_count", 0)
            oh = last_fp.get(f"{split}_hash",  "")
            nh = current_fp.get(f"{split}_hash",  "")
            if oc != nc:
                changes.append(f"{split}: {oc}→{nc}장 (+{nc-oc})")
            elif oh != nh:
                changes.append(f"{split}: 파일 교체됨")
        return (True, " / ".join(changes)) if changes else (False, "변경 없음")

    # ── 사용 가능한 체크포인트 목록 ──────────────────────────────────────
    def list_checkpoints(self, runs_dir: Path) -> list[dict]:
        """runs/ 하위에서 last.pt 체크포인트 목록 반환"""
        result = []
        for last_pt in sorted(runs_dir.glob("*/weights/last.pt"),
                              key=lambda p: p.stat().st_mtime, reverse=True):
            run_name = last_pt.parent.parent.name
            best_pt  = last_pt.parent / "best.pt"
            csv_path = last_pt.parent.parent / "results.csv"
            epoch = "?"
            if csv_path.exists():
                try:
                    df = pd.read_csv(csv_path)
                    epoch = str(len(df))
                except Exception:
                    pass
            result.append({
                "run_name":  run_name,
                "last_pt":   str(last_pt),
                "has_best":  best_pt.exists(),
                "epochs_done": epoch,
                "mtime":     datetime.fromtimestamp(last_pt.stat().st_mtime).strftime("%m/%d %H:%M"),
            })
        return result

    # ── 등록 ──────────────────────────────────────────────────────────────
    def register(
        self,
        version: str,
        run_name: str,
        train_mode: str,
        base_model: str,
        checkpoint_src: str,
        hyper: dict,
        data_counts: dict,
        data_fingerprint: dict,
        src_pt: Path,
        metrics: dict,
    ) -> dict:
        versioned_pt = self.versions_dir / f"{run_name}.pt"
        shutil.copy2(src_pt, versioned_pt)

        # fitness 계산 (metrics 안에 미리 들어 있어도 재계산)
        score = metrics.get("fitness", self.fitness(metrics))
        metrics["fitness"] = round(float(score), 4)

        prev_best = self._data.get("best_fitness", 0.0)
        is_best   = score > prev_best

        entry = {
            "version":          version,
            "run_name":         run_name,
            "timestamp":        datetime.now().isoformat(),
            "train_mode":       train_mode,
            "model_arch":       "yolov8s",
            "base_model":       base_model,
            "checkpoint_src":   checkpoint_src,
            "git_commit":       self.git_commit(),
            "env":              self.env_snapshot(),
            "hyper":            hyper,
            "data":             data_counts,
            "data_fingerprint": data_fingerprint,
            "metrics":          metrics,
            "pt_file":          str(versioned_pt.relative_to(BASE_DIR)),
            "is_best":          is_best,
        }

        if is_best:
            for v in self._data["versions"]:
                v["is_best"] = False
            self._data["best_version"] = version
            self._data["best_fitness"] = round(score, 4)

        self._data["versions"].append(entry)
        self._save()
        return entry

    # ── best PT 경로 ─────────────────────────────────────────────────────
    def best_pt_path(self) -> Path | None:
        bv = self._data.get("best_version")
        if bv is None:
            return None
        for v in self._data["versions"]:
            if v["version"] == bv:
                return BASE_DIR / v["pt_file"]
        return None

    # ── 이력 출력 ─────────────────────────────────────────────────────────
    def print_history(self):
        vs = self._data["versions"]
        if not vs:
            print("  (기록 없음)")
            return
        print(f"  {'Ver':<5} {'Mode':<10} {'Train':>6} {'Fitness':>8} {'mAP50':>7}"
              f" {'mAP5095':>8} {'P':>7} {'R':>7} {'F1':>7} {'FPS':>6}  비고")
        print("  " + "-" * 105)
        for v in sorted(vs,
                        key=lambda x: x["metrics"].get("fitness",
                                       0.1 * x["metrics"].get("map50",0) + 0.9 * x["metrics"].get("map5095",0)),
                        reverse=True):
            m  = v["metrics"]
            dc = v.get("data", {})
            fit = m.get("fitness", 0.1*m.get("map50",0) + 0.9*m.get("map5095",0))
            flag = " ◀ BEST" if v["is_best"] else ""
            print(
                f"  {v['version']:<5} {v.get('train_mode','new'):<10}"
                f" {dc.get('train',0):>6}"
                f" {fit:>8.4f}"
                f" {m.get('map50',0):>7.4f} {m.get('map5095',0):>8.4f}"
                f" {m.get('precision',0):>7.4f} {m.get('recall',0):>7.4f}"
                f" {m.get('f1',0):>7.4f} {m.get('fps',0):>6.1f}"
                f"  {flag}"
            )


registry = ModelRegistry(REGISTRY, VERS_DIR)
print(f"[ModelRegistry] 기존 버전 {len(registry._data['versions'])}개")
if registry._data["best_version"]:
    print(f"  현재 best: {registry._data['best_version']}  fitness={registry._data.get('best_fitness',0)}")


## 1. 데이터셋 준비 — Roboflow Universe 자동 다운로드

5개 클래스(`plate / tissue / smartphone / remote / glasses`)에 대해 Roboflow Universe 의 외부 데이터셋을 자동 다운로드 + 우리 5-class 구조로 병합한다.

### 사전 준비
- Roboflow 계정: https://app.roboflow.com (무료)
- API Key: Settings → Workspace → Roboflow API → Private API Key

### 데이터셋 출처
| 클래스 | Workspace / Project | 비고 |
|--------|---------------------|------|
| plate      | `ata-ghofg/bowls-vs-plates`     | bowl 클래스는 drop |
| tissue     | `mymap/-tissue-quality`         | classification 데이터셋 — detection 다운 불가 (스킵) |
| smartphone | `pryexam/smart-phone-jyvdu`     | 외부 클래스명 `celular` (포르투갈어) → smartphone 매핑 |
| remote     | `data-sets-ufvph/remote-as3zm`  | export 버전 미공개 — 다운 불가 (스킵) |
| glasses    | `khaled-qwwx7/glasses-kwmgc`    | sunglasses 는 drop |

> ⚠ tissue, remote 는 외부 데이터셋이 적합하지 않아 **자체 캡처 보강 필요**.  
> 다른 데이터셋으로 교체하려면 아래 셀의 `ROBOFLOW_DATASETS` 의 `workspace/project/version` 만 수정.

### 직접 수집하는 경우 (수동)
```
data/images/{train,valid,test}/   data/labels/{train,valid,test}/
```
라벨: YOLO 형식 `<class_id> <cx> <cy> <w> <h>` (좌표 0~1 정규화). 자세한 가이드는 `data/README.md` 참조.


In [ ]:
# ── Roboflow 데이터셋 다운로드 설정 ──────────────────────────────────────
# True 이면 다음 셀 실행 시 다운로드 + data/ 자동 병합. False 이면 수동 데이터 사용.
USE_ROBOFLOW = True

# API Key 하드코딩 (개인용 워크스페이스에서만 권장. 공개 repo 라면 환경변수 사용)
# 환경변수 ROBOFLOW_API_KEY 가 있으면 그쪽이 우선.
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "QRS5eoqbrH6kj3tX9cEQ")

# 다운로드 캐시 위치 (data/ 와 분리 — 재실행 시 캐시 재사용)
ROBOFLOW_CACHE = BASE_DIR / "_roboflow_cache"

# 5개 데이터셋 정의 — workspace/project/version + 외부클래스명 매핑
# version=None 이면 가장 최신 버전 자동 선택
ROBOFLOW_DATASETS = [
    {
        "tag":       "plate",
        "workspace": "ata-ghofg",
        "project":   "bowls-vs-plates",
        "version":   3,
        "class_map": {
            "plate":  "plate",
            "plates": "plate",
            "bowl":   None,         # drop
            "bowls":  None,
        },
    },
    {
        "tag":       "tissue",
        "workspace": "mymap",
        "project":   "-tissue-quality",
        "version":   1,
        # ⚠ 이 데이터셋은 classification 형식이라 detection 다운 불가.
        #   별도 detection tissue 데이터셋으로 교체하거나 자체 캡처 사용.
        "class_map": {
            "tissue":      "tissue",
            "tissue_box":  "tissue",
            "toilet_paper":"tissue",
            "kleenex":     "tissue",
        },
    },
    {
        "tag":       "smartphone",
        "workspace": "pryexam",
        "project":   "smart-phone-jyvdu",
        "version":   2,
        "class_map": {
            "smart-phone": "smartphone",
            "smartphone":  "smartphone",
            "smart_phone": "smartphone",
            "phone":       "smartphone",
            "mobile":      "smartphone",
            "celular":     "smartphone",   # 포르투갈/스페인어 — 이 데이터셋의 라벨
            "movil":       "smartphone",
            "telefono":    "smartphone",
        },
    },
    {
        "tag":       "remote",
        "workspace": "data-sets-ufvph",
        "project":   "remote-as3zm",
        "version":   None,             # 자동 탐색 (export 버전 미공개일 수 있음)
        "class_map": {
            "remote":         "remote",
            "remote_control": "remote",
            "remotecontrol":  "remote",
            "tv-remote":      "remote",
            "tv_remote":      "remote",
        },
    },
    {
        "tag":       "glasses",
        "workspace": "khaled-qwwx7",
        "project":   "glasses-kwmgc",
        "version":   2,
        "class_map": {
            "glasses":     "glasses",
            "eyeglasses":  "glasses",
            "sunglasses":  None,
        },
    },
]

print(f"USE_ROBOFLOW    = {USE_ROBOFLOW}")
print(f"API key         = {'(설정됨)' if ROBOFLOW_API_KEY else '(미설정)'}")
print(f"등록 데이터셋    = {len(ROBOFLOW_DATASETS)}개")
print(f"캐시 위치        = {ROBOFLOW_CACHE}")


In [ ]:
# Roboflow 다운로드 + data/ 자동 병합 (USE_ROBOFLOW=True 시 실행)

def _download_and_merge():
    import re
    try:
        from roboflow import Roboflow
    except ImportError:
        print("[ERROR] roboflow 미설치 — `pip install roboflow` 실행 필요")
        return

    if not ROBOFLOW_API_KEY:
        print("[ERROR] ROBOFLOW_API_KEY 미설정")
        return

    NAME2ID = {n: i for i, n in enumerate(CLASS_NAMES)}

    # 디렉토리 보장
    for split in ["train", "valid", "test"]:
        (DATA_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (DATA_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
    ROBOFLOW_CACHE.mkdir(parents=True, exist_ok=True)

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    def latest_version(project) -> int | None:
        try:
            vs = project.versions()
            nums = []
            for v in vs:
                m = re.search(r"/(\d+)$", str(v.id))
                if m:
                    nums.append(int(m.group(1)))
            return max(nums) if nums else None
        except Exception:
            return None

    def download(ds_def):
        tag = ds_def["tag"]
        cache = ROBOFLOW_CACHE / tag
        # 캐시 활용 — 이미 다운로드된 게 있으면 그대로 사용
        if cache.exists():
            sub = next((p for p in cache.iterdir() if p.is_dir()), None)
            if sub and (sub / "data.yaml").exists():
                print(f"  캐시 사용: {sub.name}")
                return sub
        try:
            proj = rf.workspace(ds_def["workspace"]).project(ds_def["project"])
        except Exception as e:
            print(f"  [ERROR] 프로젝트 접근 실패: {e}")
            return None
        ver = ds_def.get("version") or latest_version(proj)
        if ver is None:
            print(f"  [ERROR] 사용 가능한 버전 없음 — 스킵")
            return None
        cache.mkdir(parents=True, exist_ok=True)
        cwd = os.getcwd()
        try:
            os.chdir(cache)
            ds = proj.version(ver).download("yolov8")
            return Path(ds.location)
        except Exception as e:
            print(f"  [ERROR] 다운로드 실패 (v{ver}): {e}")
            return None
        finally:
            os.chdir(cwd)

    def remap_label(src_lbl: Path, dst_lbl: Path, ext_id_to_our: dict) -> int:
        if not src_lbl.exists():
            dst_lbl.write_text("")
            return 0
        out = []
        for line in src_lbl.read_text().splitlines():
            p = line.strip().split()
            if len(p) < 5:
                continue
            try:
                cid = int(p[0])
            except ValueError:
                continue
            new_id = ext_id_to_our.get(cid)
            if new_id is None:
                continue
            out.append(f"{new_id} {' '.join(p[1:5])}")
        dst_lbl.write_text("\n".join(out))
        return len(out)

    def merge(downloaded: Path, ds_def: dict):
        tag = ds_def["tag"]
        ycfg = yaml.safe_load((downloaded / "data.yaml").read_text())
        ext_names = ycfg.get("names", [])
        if isinstance(ext_names, dict):
            ext_names = [ext_names[k] for k in sorted(ext_names.keys())]
        cmap = {k.lower(): v for k, v in ds_def["class_map"].items()}
        ext_id_to_our = {}
        for i, name in enumerate(ext_names):
            nm = name.lower().strip().replace(" ", "_")
            target = cmap.get(nm)
            if target is None and nm not in cmap:
                for k, v in cmap.items():
                    if k in nm or nm in k:
                        target = v
                        break
            if target:
                ext_id_to_our[i] = NAME2ID[target]
        print(f"  외부 클래스: {ext_names}  →  매핑: {ext_id_to_our}")
        if not ext_id_to_our:
            print(f"  [WARN] 매핑 없음 — 스킵")
            return

        for src_split in ["train", "valid", "test"]:
            sid = downloaded / src_split / "images"
            sld = downloaded / src_split / "labels"
            if not sid.exists():
                continue
            did = DATA_DIR / "images" / src_split
            dld = DATA_DIR / "labels" / src_split
            n_img = n_lbl = 0
            for img in sorted(sid.iterdir()):
                if img.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                new_name = f"{tag}_{img.name}"
                shutil.copy2(img, did / new_name)
                n_img += 1
                n_lbl += remap_label(
                    sld / (img.stem + ".txt"),
                    dld / (Path(new_name).stem + ".txt"),
                    ext_id_to_our,
                )
            print(f"  {src_split:5s}: {n_img:5d} 장,  {n_lbl:5d} 라벨")

    for ds_def in ROBOFLOW_DATASETS:
        print(f"\n[{ds_def['tag']}] {ds_def['workspace']}/{ds_def['project']}")
        d = download(ds_def)
        if d is not None:
            merge(d, ds_def)

    # 최종 통계
    print("\n" + "=" * 70)
    print("  data/ 최종 분포")
    print("=" * 70)
    for split in ["train", "valid", "test"]:
        n_img = len(list((DATA_DIR / "images" / split).glob("*.[jp][pn]g")))
        cls = [0] * len(CLASS_NAMES)
        for lbl in (DATA_DIR / "labels" / split).glob("*.txt"):
            for line in lbl.read_text().splitlines():
                p = line.split()
                if len(p) >= 5:
                    try:
                        cls[int(p[0])] += 1
                    except (ValueError, IndexError):
                        pass
        print(f"\n  [{split}] images={n_img}")
        for i, n in enumerate(CLASS_NAMES):
            flag = "  ✗ 데이터 없음 — 자체 캡처/대체 필요" if cls[i] == 0 else ""
            print(f"    {i} {n:<12} {cls[i]:>5}{flag}")


if USE_ROBOFLOW:
    _download_and_merge()
else:
    print("USE_ROBOFLOW=False — 다운로드 스킵 (수동 데이터 사용)")


## 2. YAML 파일 생성

In [ ]:
yaml_content = {
    "path":  str(DATA_DIR.resolve()),
    "train": "images/train",
    "val":   "images/valid",
    "test":  "images/test",
    "nc":    len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_content, f, allow_unicode=True, default_flow_style=False)

print(f"YAML 저장: {YAML_PATH}")
print(YAML_PATH.read_text())

## 3. 데이터 검증 및 클래스 분포

In [ ]:
def validate_split(split: str) -> dict:
    img_dir = DATA_DIR / "images" / split
    lbl_dir = DATA_DIR / "labels" / split
    imgs  = sorted(img_dir.glob("*.[jp][pn]g"))
    counter, no_lbl, bad_ann = Counter(), [], []
    for img in imgs:
        lbl = lbl_dir / (img.stem + ".txt")
        if not lbl.exists():
            no_lbl.append(img.name); continue
        for line in lbl.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) != 5: bad_ann.append(img.name); break
            cls_id = int(parts[0])
            if 0 <= cls_id < len(CLASS_NAMES):
                counter[CLASS_NAMES[cls_id]] += 1
    lbls = sorted(lbl_dir.glob("*.txt"))
    return {"n_img": len(imgs), "n_lbl": len(lbls),
            "counter": counter, "no_lbl": no_lbl, "bad_ann": bad_ann}

print("=" * 60)
all_stats = {}
for split in ["train", "valid", "test"]:
    s = validate_split(split)
    all_stats[split] = s
    ok = s["n_img"] == s["n_lbl"]
    print(f"  [{split}]  images={s['n_img']}  labels={s['n_lbl']}  {'OK' if ok else 'MISMATCH'}")
    if s["no_lbl"]: print(f"    [WARN] 라벨 없는 이미지 {len(s['no_lbl'])}개")
    if s["bad_ann"]: print(f"    [WARN] 형식 오류 라벨 {len(s['bad_ann'])}개")
    max_cnt = max(s["counter"].values(), default=1)
    for cls in CLASS_NAMES:
        cnt = s["counter"].get(cls, 0)
        bar = "█" * int(cnt / max_cnt * 25)
        print(f"    {cls:<16} {bar:<25} {cnt}")
print("=" * 60)

train_cnt = all_stats["train"]["counter"]
if train_cnt:
    mx, mn = max(train_cnt.values()), min(train_cnt.values())
    if mn > 0 and mx / mn > 5:
        print(f"[WARN] 클래스 불균형 {mx/mn:.1f}x — 추가 수집 고려")

## 4. 데이터 시각화

In [ ]:
def show_samples(split: str = "train", n: int = 6):
    img_dir = DATA_DIR / "images" / split
    lbl_dir = DATA_DIR / "labels" / split
    imgs = sorted(img_dir.glob("*.[jp][pn]g"))[:n]
    if not imgs: print(f"{split} 이미지 없음"); return

    cols = min(3, len(imgs))
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ih, iw = img.shape[:2]
        ax.imshow(img)
        lbl = lbl_dir / (img_path.stem + ".txt")
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5: continue
                cid = int(parts[0])
                cx, cy, nw, nh = map(float, parts[1:])
                x1, y1 = (cx - nw/2)*iw, (cy - nh/2)*ih
                color = COLORS[cid % len(COLORS)]
                ax.add_patch(patches.Rectangle(
                    (x1, y1), nw*iw, nh*ih, lw=2, ec=color, fc="none"))
                ax.text(x1, y1-3, CLASS_NAMES[cid], fontsize=8,
                        color=color, weight="bold")
        ax.set_title(img_path.name[:28], fontsize=8); ax.axis("off")
    for ax in axes[len(imgs):]: ax.axis("off")
    plt.suptitle(f"[{split}] 샘플 {len(imgs)}장", fontsize=12, y=1.01)
    plt.tight_layout(); plt.show()

show_samples("train", n=6)

## 5. 모델 학습

**YOLOv8s + 5-class + 소규모 데이터 + top-down RealSense 도메인** 에 맞춘 하이퍼파라미터.

| 핵심 설정 | 값 | 근거 |
|-----------|----|----|
| `freeze=10` | backbone 동결 | <400장 + COCO 인접 도메인은 backbone freeze 가 과적합 억제에 가장 효과적 |
| `epochs=100` / `patience=20` | 학습 길이 | 소규모 데이터는 60~80 epoch 에 plateau, 150 은 과함 |
| `batch=-1` (auto) | VRAM 자동 활용 | YOLOv8 auto-batch 가 GPU 메모리 60% 자동 할당 |
| `cache='ram'` | 데이터 캐싱 | 소규모 데이터 → epoch 시간 -10~30% |
| `seed=42` / `deterministic=True` | 재현성 | 버전 비교 신뢰성 보장 |
| `lr0=0.0005` (AdamW) | freeze 시 head 만 학습 | 기본 0.001 보다 낮게 — 안정 fine-tune |
| `flipud=0.2` / `shear=0` / `degrees=15` | 도메인 정합 | top-down 평행 광학 + 로봇 yaw 변동 범위 |
| `mosaic=0.7` / `mixup=0.2` | 합성 증강 | 5 클래스에서 mosaic=1.0 라벨 과밀, mixup 일반화 |
| `copy_paste=0.0` | detect 모드 | segment 전용, detect 효과 미미 |
| `label_smoothing=0.0` | calibration 보존 | 로봇이 conf 로 grasp 결정 → 신뢰도 왜곡 방지 |
| `close_mosaic=15` | 마지막 안정화 | 마지막 15 epoch 모자이크 OFF → 단일 이미지 fine-tune |
| `save_period=20` | checkpoint | 20 epoch 마다 `epoch{N}.pt` 저장 (resume / more_data 대응) |


In [ ]:
# ── 학습 모드 ──────────────────────────────────────────────────────────────
#
#  "new"        : BASE_MODEL 에서 새 학습 시작
#  "resume"     : 중단된 학습의 last.pt 에서 재개 (같은 데이터, 버전 번호 유지)
#  "more_data"  : 데이터가 늘었을 때 이전 best.pt 를 시작점으로 새 버전 학습
#
TRAIN_MODE = "new"

# "resume" 시 — 재개할 run 이름 (RUNS_DIR/{RESUME_RUN_NAME}/weights/last.pt 필요)
RESUME_RUN_NAME = ""     # 예: "v2_cobot_0504_1430"

# "more_data" 시 — 시작점 PT 경로. None 이면 현재 registry best 자동 선택
MORE_DATA_BASE_PT = None  # 예: str(BASE_DIR / "versions" / "v2_cobot_0504_1430.pt")

# ── 베이스 모델 (TRAIN_MODE="new" 시) ─────────────────────────────────────
BASE_MODEL = "yolov8s.pt"   # YOLOv8 small, COCO pretrained

# ── 하이퍼파라미터 (YOLOv8s + 5-class + 소규모 데이터 + top-down RealSense) ──
#
# 설계 근거:
#   - freeze=10            : <400장 + COCO 인접 도메인(필통/블록/접시/후추통)에서
#                            backbone 동결이 과적합 억제에 가장 효과적 (Ultralytics 공식 가이드)
#   - epochs=100/patience=20: <400장 기준 plateau 가 60~80 epoch 에 도달, 150 은 과함
#   - batch=-1             : auto-batch 로 GPU VRAM 60% 자동 활용
#   - cache="ram"          : 소규모 데이터 → RAM 캐시로 epoch 시간 -10~30%
#   - seed=42, deterministic=True : 버전 비교의 재현성
#   - flipud=0.2, shear=0, degrees=15 : top-down 시점 도메인 정합 (조명 비대칭/평행 광학계)
#   - mosaic=0.7, mixup=0.2 : 5 클래스에서 mosaic=1.0 은 라벨 과밀, mixup 은 일반화 향상
#   - copy_paste=0.0       : detect 모드에서 효과 미미 (segment 전용)
#   - label_smoothing=0.0  : 5 클래스 + calibration 보존 (로봇이 conf 로 grasp 결정)
HYPER = {
    # ── 기본 학습 ──
    "epochs":          100,
    "patience":        20,
    "batch":           -1,        # auto-batch
    "imgsz":           640,
    "workers":         4,
    "cache":           "ram",     # 소규모 데이터 RAM 캐시
    "seed":            42,
    "deterministic":   True,

    # ── Optimizer / Scheduler ──
    "optimizer":       "AdamW",
    "lr0":             0.0005,    # freeze 시 head 만 학습 → 기본보다 낮게
    "lrf":             0.01,
    "weight_decay":    0.0005,
    "warmup_epochs":   3,
    "warmup_momentum": 0.8,
    "cos_lr":          True,

    # ── Transfer learning ──
    "freeze":          10,        # backbone(layer 0~9) 동결, head 만 fine-tune

    # ── 색상 증강 ──
    "hsv_h":           0.015,     # YOLO 기본 (4클래스 색상 보존)
    "hsv_s":           0.5,
    "hsv_v":           0.4,

    # ── 기하 증강 (top-down 도메인 정합) ──
    "degrees":         15.0,      # 로봇 yaw 변동 범위
    "translate":       0.1,
    "scale":           0.5,
    "shear":           0.0,       # top-down 평행 광학에서 shear 비현실적
    "perspective":     0.0,
    "flipud":          0.2,       # 조명 비대칭 고려 약하게
    "fliplr":          0.5,

    # ── 합성 증강 ──
    "mosaic":          0.7,       # 5 클래스 + 소규모에서 1.0 은 라벨 과밀
    "mixup":           0.2,       # 일반화 향상
    "copy_paste":      0.0,       # detect 모드 효과 미미
    "close_mosaic":    15,        # 마지막 15 epoch 은 mosaic OFF (안정 fine-tune)

    # ── Regularization ──
    "label_smoothing": 0.0,       # calibration 보존
    "dropout":         0.0,

    # ── 저장 / 로깅 ──
    "save_period":     20,        # runs/.../weights/epoch{N}.pt 저장 간격
    "plots":           True,
    "verbose":         True,
    "amp":             True,      # mixed precision
}

print(f"TRAIN_MODE = {TRAIN_MODE}")
print(f"BASE_MODEL = {BASE_MODEL}")
print(f"freeze     = {HYPER['freeze']} (backbone 고정)")
print(f"epochs     = {HYPER['epochs']}  patience = {HYPER['patience']}")
print(f"checkpoint 저장 간격 = {HYPER['save_period']} epoch")
print(f"seed       = {HYPER['seed']} (재현성 보장)")


In [ ]:
# ── 체크포인트 & 데이터 상태 확인 ─────────────────────────────────────────
# 학습 전 현재 상황 파악. TRAIN_MODE 선택과 RESUME_RUN_NAME 설정에 참고한다.

current_fp           = ModelRegistry.data_fingerprint(DATA_DIR)
data_changed, change_msg = registry.detect_data_change(current_fp)

print("=" * 65)
print("  [현재 데이터 상태]")
for split in ["train", "valid", "test"]:
    cnt = current_fp[f"{split}_count"]
    h   = current_fp[f"{split}_hash"]
    print(f"    {split:<6}: {cnt:>4}장  (hash={h})")
print(f"  데이터 변경 감지: {change_msg}")
print()

# ── 재개 가능한 체크포인트 목록 ───────────────────────────────────────────
ckpts = registry.list_checkpoints(RUNS_DIR)
if ckpts:
    print("  [재개 가능한 체크포인트 (last.pt)]")
    print(f"    {'Run Name':<30} {'Epoch':>6}  시각")
    print("    " + "-" * 55)
    for ck in ckpts:
        print(f"    {ck['run_name']:<30} {ck['epochs_done']:>6}  {ck['mtime']}")
    print()
    print(f"  ※ TRAIN_MODE='resume' 시: RESUME_RUN_NAME = \"{ckpts[0]['run_name']}\"")
else:
    print("  [재개 가능한 체크포인트] 없음")
print()

# ── 현재 best 정보 ────────────────────────────────────────────────────────
best_pt = registry.best_pt_path()
if best_pt:
    print(f"  [현재 best] {registry._data['best_version']}  "
          f"mAP50={registry._data['best_map50']}  ({best_pt.name})")
    print(f"  ※ TRAIN_MODE='more_data' 시 이 PT 를 시작점으로 자동 사용")
else:
    print("  [현재 best] 없음 (최초 학습)")
print("=" * 65)

# ── 모드 자동 추천 ────────────────────────────────────────────────────────
if ckpts and not data_changed:
    print(f"\n  ▶ 추천 TRAIN_MODE = 'resume'  ({ckpts[0]['run_name']} 재개)")
elif data_changed:
    print(f"\n  ▶ 추천 TRAIN_MODE = 'more_data'  (데이터 변경: {change_msg})")
else:
    print(f"\n  ▶ 추천 TRAIN_MODE = 'new'")

In [ ]:
from ultralytics import YOLO

# ── 데이터 fingerprint & 변경 감지 ───────────────────────────────────────
current_fp           = ModelRegistry.data_fingerprint(DATA_DIR)
data_changed, change_msg = registry.detect_data_change(current_fp)
n_train = current_fp["train_count"]
n_valid = current_fp["valid_count"]
n_test  = current_fp["test_count"]

if n_train == 0:
    raise RuntimeError(f"train 이미지 없음: {DATA_DIR / 'images' / 'train'}")
if n_valid == 0:
    raise RuntimeError("valid 이미지 없음")

print(f"데이터: train={n_train}  valid={n_valid}  test={n_test}")
print(f"데이터 변경: {change_msg}")

# ── TRAIN_MODE 분기 ────────────────────────────────────────────────────────
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if TRAIN_MODE == "resume":
    if not RESUME_RUN_NAME:
        raise ValueError(
            "RESUME_RUN_NAME 을 지정하세요.\n"
            "위의 체크포인트 확인 셀에서 run 이름을 확인하세요."
        )
    last_pt = RUNS_DIR / RESUME_RUN_NAME / "weights" / "last.pt"
    if not last_pt.exists():
        raise FileNotFoundError(
            f"last.pt 없음: {last_pt}\n"
            f"  가용 체크포인트: {[c['run_name'] for c in registry.list_checkpoints(RUNS_DIR)]}"
        )
    VERSION        = RESUME_RUN_NAME.split("_")[0]
    RUN_NAME       = RESUME_RUN_NAME
    checkpoint_src = str(last_pt)
    model          = YOLO(str(last_pt))
    resume_flag    = True
    print(f"\n[RESUME] version={VERSION}  from={last_pt}")

elif TRAIN_MODE == "more_data":
    if not data_changed:
        print("[WARN] 데이터 변경이 감지되지 않음 — 계속 진행합니다.")
    base_pt = (Path(MORE_DATA_BASE_PT) if MORE_DATA_BASE_PT
               else registry.best_pt_path())
    if base_pt is None:
        raise FileNotFoundError(
            "시작점 PT 없음. 먼저 TRAIN_MODE=\'new\' 로 최초 학습을 실행하세요."
        )
    if not base_pt.exists():
        raise FileNotFoundError(f"PT 파일 없음: {base_pt}")
    VERSION        = registry.next_version()
    RUN_NAME       = registry.make_run_name(VERSION)
    checkpoint_src = str(base_pt)
    model          = YOLO(str(base_pt))
    resume_flag    = False
    print(f"\n[MORE_DATA] version={VERSION}  warm-start={base_pt.name}")
    print(f"  데이터 변화: {change_msg}")

else:  # "new"
    VERSION        = registry.next_version()
    RUN_NAME       = registry.make_run_name(VERSION)
    checkpoint_src = BASE_MODEL
    model          = YOLO(BASE_MODEL)
    resume_flag    = False
    print(f"\n[NEW] version={VERSION}  base={BASE_MODEL}")

print(f"run_name={RUN_NAME}\n")

# ── 학습 실행 ──────────────────────────────────────────────────────────────
# 체크포인트 파일 위치:
#   runs/{RUN_NAME}/weights/epoch{N}.pt  — save_period 배수 epoch 마다
#   runs/{RUN_NAME}/weights/last.pt      — 매 epoch 갱신 (중단 재개용)
#   runs/{RUN_NAME}/weights/best.pt      — val mAP50 기준 최고 epoch
results = model.train(
    data=str(YAML_PATH),
    project=str(RUNS_DIR),
    name=RUN_NAME,
    resume=resume_flag,
    exist_ok=(TRAIN_MODE == "resume"),
    device=0 if torch.cuda.is_available() else "cpu",
    **HYPER,
)

TRAINED_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
if not TRAINED_PT.exists():
    raise FileNotFoundError(
        f"best.pt 없음: {TRAINED_PT}\n"
        "  last.pt 를 대신 사용하려면 TRAINED_PT 를 last.pt 로 수정하세요."
    )

ckpt_files = sorted((RUNS_DIR / RUN_NAME / "weights").glob("epoch*.pt"))
print(f"\n[학습 완료]")
print(f"  best.pt  : {TRAINED_PT}")
print(f"  last.pt  : {RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'}")
if ckpt_files:
    print(f"  체크포인트: {[f.name for f in ckpt_files]}")

## 6. 평가 & 버전 등록 & best.pt 자동 갱신

Test set 지표를 측정하고 `model_registry.json` 에 기록한다.  
**mAP@50** 기준으로 역대 최고이면 `runs/best.pt` 와 `object_detection/resource/best.pt` 를 덮어쓴다.

In [ ]:
if not TRAINED_PT.exists():
    raise FileNotFoundError(f"학습된 PT 없음: {TRAINED_PT}")

eval_model = YOLO(str(TRAINED_PT))

# ── Test set 평가 (plots=True 로 confusion matrix / PR / F1 curve 자동 생성) ──
metrics = eval_model.val(
    data=str(YAML_PATH),
    split="test",
    imgsz=HYPER["imgsz"],
    batch=8,
    verbose=False,
    plots=True,                # ← confusion_matrix.png, PR_curve.png 등 생성
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_test_eval",
    exist_ok=True,
)
box = metrics.box
mp  = float(box.mp)
mr  = float(box.mr)
f1  = 2 * mp * mr / (mp + mr + 1e-9)

per_class_ap = {
    CLASS_NAMES[int(idx)]: round(float(ap), 4)
    for idx, ap in zip(box.ap_class_index, box.ap50)
    if int(idx) < len(CLASS_NAMES)
}

# ── FPS 측정 (실제 test 이미지로 — NMS 비용 반영) ─────────────────────────
test_imgs = sorted((DATA_DIR / "images" / "test").glob("*.[jp][pn]g"))
if test_imgs:
    sample_imgs = [cv2.imread(str(p)) for p in test_imgs[:5]]
    # warm-up 30 회
    for _ in range(30):
        eval_model(sample_imgs[0], verbose=False)
    t_list = []
    for _ in range(50):
        img = sample_imgs[_ % len(sample_imgs)]
        t0 = time.perf_counter()
        eval_model(img, verbose=False)
        t_list.append(time.perf_counter() - t0)
    fps = 1.0 / (sum(t_list) / len(t_list))
else:
    fps = 0.0
    print("[WARN] test 이미지 없음 — FPS 측정 스킵")

# ── center-point 오차 (로봇 파지 직결 지표) ───────────────────────────────
def compute_center_error(model, imgs_dir: Path, labels_dir: Path,
                         conf: float = 0.25) -> tuple[float, int]:
    """GT bbox 와 매칭된 예측의 center 픽셀 오차 평균. (mean_err_px, matched_pairs)"""
    errs = []
    for img_path in sorted(imgs_dir.glob("*.[jp][pn]g")):
        lbl_path = labels_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]
        gts = []
        for line in lbl_path.read_text().strip().split("\n"):
            if not line.strip():
                continue
            parts = line.split()
            cls = int(parts[0])
            cx, cy = float(parts[1]) * W, float(parts[2]) * H
            gts.append((cls, cx, cy))
        if not gts:
            continue
        res = model(img, conf=conf, verbose=False)[0]
        if res.boxes is None or len(res.boxes) == 0:
            continue
        preds = []
        for b in res.boxes:
            x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
            preds.append((int(b.cls[0]), (x1+x2)/2, (y1+y2)/2))
        # 같은 클래스끼리 가장 가까운 GT-pred 매칭 (greedy)
        used = set()
        for g_cls, gx, gy in gts:
            best_d, best_i = None, -1
            for i, (p_cls, px, py) in enumerate(preds):
                if i in used or p_cls != g_cls:
                    continue
                d = ((gx-px)**2 + (gy-py)**2) ** 0.5
                if best_d is None or d < best_d:
                    best_d, best_i = d, i
            if best_i >= 0:
                used.add(best_i)
                errs.append(best_d)
    if not errs:
        return 0.0, 0
    return float(sum(errs) / len(errs)), len(errs)

center_err_px, n_matched = compute_center_error(
    eval_model,
    DATA_DIR / "images" / "test",
    DATA_DIR / "labels" / "test",
)

measured_metrics = {
    "map50":          round(float(box.map50), 4),
    "map5095":        round(float(box.map),   4),
    "precision":      round(mp, 4),
    "recall":         round(mr, 4),
    "f1":             round(f1, 4),
    "fps":            round(fps, 1),
    "center_err_px":  round(center_err_px, 2),
    "center_matches": n_matched,
    "per_class_ap50": per_class_ap,
}
# fitness 계산 (best 선정 기준)
measured_metrics["fitness"] = round(
    0.1 * measured_metrics["map50"] + 0.9 * measured_metrics["map5095"], 4
)

# ── 레지스트리 등록 ────────────────────────────────────────────────────────
entry = registry.register(
    version          = VERSION,
    run_name         = RUN_NAME,
    train_mode       = TRAIN_MODE,
    base_model       = BASE_MODEL if TRAIN_MODE == "new" else checkpoint_src,
    checkpoint_src   = checkpoint_src,
    hyper            = HYPER,
    data_counts      = {"train": n_train, "valid": n_valid, "test": n_test},
    data_fingerprint = current_fp,
    src_pt           = TRAINED_PT,
    metrics          = measured_metrics,
)

# ── 결과 출력 ──────────────────────────────────────────────────────────────
print("=" * 60)
print(f"  [{VERSION}] {RUN_NAME}  mode={TRAIN_MODE}")
print(f"  ★ Fitness    : {measured_metrics['fitness']:.4f}   ← best 선정 기준")
print(f"  mAP@50       : {measured_metrics['map50']:.4f}")
print(f"  mAP@50-95    : {measured_metrics['map5095']:.4f}")
print(f"  Precision    : {measured_metrics['precision']:.4f}")
print(f"  Recall       : {measured_metrics['recall']:.4f}")
print(f"  F1           : {measured_metrics['f1']:.4f}")
print(f"  FPS          : {measured_metrics['fps']:.1f}")
print(f"  center error : {center_err_px:.2f} px  ({n_matched} matched)")
print()
print("  [클래스별 AP@50]")
for cls, ap in sorted(per_class_ap.items(), key=lambda x: -x[1]):
    bar  = "█" * int(ap * 30)
    flag = "  ⚠" if ap < 0.5 else ""
    print(f"    {cls:<18} {ap:.4f}  {bar}{flag}")
print("=" * 60)

# 공식 plot 위치 안내
plot_dir = RUNS_DIR / f"{RUN_NAME}_test_eval"
print(f"\n  공식 평가 플롯: {plot_dir}")
print(f"    - confusion_matrix.png / confusion_matrix_normalized.png")
print(f"    - PR_curve.png  P_curve.png  R_curve.png  F1_curve.png")
print(f"    - val_batch*_pred.jpg / val_batch*_labels.jpg")

# ── best.pt 자동 갱신 ─────────────────────────────────────────────────────
if entry["is_best"]:
    best_in_runs = RUNS_DIR / "best.pt"
    shutil.copy2(TRAINED_PT, best_in_runs)
    print(f"\n★ NEW BEST  fitness={measured_metrics['fitness']}  ({VERSION})")
    print(f"  runs/best.pt 갱신 → {best_in_runs}")

    if OD_RESOURCE.exists():
        shutil.copy2(TRAINED_PT, OD_RESOURCE / "best.pt")
        (OD_RESOURCE / "class_name_tool.json").write_text(
            json.dumps({str(i): n for i, n in enumerate(CLASS_NAMES)},
                       indent=4, ensure_ascii=False)
        )
        print(f"  OD resource 갱신 → {OD_RESOURCE / 'best.pt'}")
    else:
        print(f"  [WARN] OD resource 경로 없음: {OD_RESOURCE}")
else:
    prev_best = registry._data.get("best_fitness", 0.0)
    diff = prev_best - measured_metrics["fitness"]
    print(f"\n  현재 best 유지 ({registry._data['best_version']}  fitness={prev_best})"
          f"  (이번 -{diff:.4f})")

print(f"\n  레지스트리 : {REGISTRY}")
print(f"  버전 PT    : {entry['pt_file']}")
print(f"  체크포인트 : {RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'}")


## 7. 학습 곡선

In [ ]:
csv_path = RUNS_DIR / RUN_NAME / "results.csv"
if not csv_path.exists():
    # 최신 run 탐색 (셀을 따로 실행하는 경우 대비)
    candidates = sorted(RUNS_DIR.glob("*/results.csv"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    csv_path = candidates[0] if candidates else None

if csv_path is None:
    print("results.csv 없음")
else:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # fitness 추이 계산 (= 0.1·mAP50 + 0.9·mAP50-95)
    if "metrics/mAP50(B)" in df.columns and "metrics/mAP50-95(B)" in df.columns:
        df["fitness"] = 0.1 * df["metrics/mAP50(B)"] + 0.9 * df["metrics/mAP50-95(B)"]

    # close_mosaic 적용 시점
    close_mosaic_ep = HYPER["epochs"] - HYPER.get("close_mosaic", 0)

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f"학습 곡선 — {csv_path.parent.name}", fontsize=13)

    def plot_col(ax, col, title, color="steelblue", mark_close=False):
        if col in df.columns:
            ax.plot(df["epoch"], df[col], color=color)
            if mark_close and close_mosaic_ep > 0:
                ax.axvline(close_mosaic_ep, color="gray", linestyle="--",
                           alpha=0.5, label="close_mosaic")
                ax.legend(fontsize=8)
            ax.set_title(title); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
        else:
            ax.text(0.5, 0.5, f"{col}\nnot found", ha="center", va="center",
                    transform=ax.transAxes)

    plot_col(axes[0,0], "train/box_loss",      "Train Box Loss",  "#e05")
    plot_col(axes[0,1], "train/cls_loss",      "Train Cls Loss",  "#e70")
    plot_col(axes[0,2], "train/dfl_loss",      "Train DFL Loss",  "#a00")
    plot_col(axes[1,0], "metrics/mAP50(B)",    "Val mAP@50",      "#080", mark_close=True)
    plot_col(axes[1,1], "metrics/mAP50-95(B)", "Val mAP@50-95",   "#048", mark_close=True)
    plot_col(axes[1,2], "fitness",             "Fitness (best 기준)", "#c06", mark_close=True)

    plt.tight_layout()
    plt.savefig(csv_path.parent / "learning_curves.png", dpi=120, bbox_inches="tight")
    plt.show()

    # best epoch 정보
    if "fitness" in df.columns:
        best_ep = df["fitness"].idxmax()
        best_row = df.iloc[best_ep]
        print(f"Best epoch (fitness): {int(best_row['epoch'])}  "
              f"fitness={best_row['fitness']:.4f}  "
              f"mAP50={best_row['metrics/mAP50(B)']:.4f}  "
              f"mAP50-95={best_row['metrics/mAP50-95(B)']:.4f}")


## 8. 클래스별 AP 바 차트

In [ ]:
rows = sorted(per_class_ap.items(), key=lambda x: -x[1])
names_plot = [r[0] for r in rows]
aps_plot   = [r[1] for r in rows]
bar_colors = ["#2ecc71" if a >= 0.7 else "#f39c12" if a >= 0.5 else "#e74c3c"
              for a in aps_plot]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(names_plot, aps_plot, color=bar_colors)
ax.set_xlim(0, 1.1)
ax.axvline(0.5, color="gray",  linestyle="--", alpha=0.6, label="0.5")
ax.axvline(0.7, color="green", linestyle="--", alpha=0.6, label="0.7")
for bar, ap in zip(bars, aps_plot):
    ax.text(ap + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{ap:.3f}", va="center", fontsize=10)
ax.set_xlabel("AP@50")
ax.set_title(f"클래스별 AP@50 — {VERSION} ({RUN_NAME})")
ax.legend()
plt.tight_layout()
plt.savefig(RUNS_DIR / RUN_NAME / "per_class_ap.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. 예측 시각화 (GT vs Pred)

In [ ]:
def visualize_predictions(model: YOLO, split: str = "test",
                           n: int = 6, conf: float = 0.25):
    img_dir = DATA_DIR / "images" / split
    lbl_dir = DATA_DIR / "labels" / split
    imgs = sorted(img_dir.glob("*.[jp][pn]g"))[:n]
    if not imgs: print(f"{split} 이미지 없음"); return

    cols = min(3, len(imgs))
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ih, iw = img.shape[:2]
        ax.imshow(img)

        lbl = lbl_dir / (img_path.stem + ".txt")
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5: continue
                cx, cy, nw, nh = map(float, parts[1:])
                x1, y1 = (cx-nw/2)*iw, (cy-nh/2)*ih
                ax.add_patch(patches.Rectangle(
                    (x1,y1), nw*iw, nh*ih, lw=2, ec="lime", fc="none"))

        for box in model(img_path, conf=conf, verbose=False)[0].boxes:
            cid   = int(box.cls)
            name  = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
            color = COLORS[cid % len(COLORS)]
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            ax.add_patch(patches.Rectangle(
                (x1,y1), x2-x1, y2-y1, lw=2, ec=color, fc="none", ls="--"))
            ax.text(x1, y1-4, f"{name} {float(box.conf):.2f}",
                    fontsize=7, color=color, weight="bold")

        ax.set_title(img_path.name[:24], fontsize=8); ax.axis("off")
    for ax in axes[len(imgs):]: ax.axis("off")
    fig.text(0.01, 0.01, "─── GT (초록)   --- 예측 (색상)", fontsize=9)
    plt.suptitle(f"예측 시각화 [{split}] {VERSION}", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(RUNS_DIR / RUN_NAME / f"pred_{split}.png",
                dpi=120, bbox_inches="tight")
    plt.show()

visualize_predictions(eval_model, split="test", n=6)

## 10. 오류 분석 (FP / FN)

In [ ]:
def _iou_xyxy(a, b):
    """단일 bbox vs bbox IoU (xyxy 픽셀 좌표)"""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih   = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter    = iw * ih
    if inter <= 0:
        return 0.0
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)


def analyze_errors(model: YOLO, split: str = "test",
                   conf: float = 0.25, iou_thr: float = 0.5):
    """IoU-matching 기반 FP/FN 분석.

    매칭 규칙:
      - GT 와 Pred 가 IoU ≥ iou_thr 이고 같은 클래스 → TP
      - 같은 IoU 안에서 클래스가 다르면 → FP(잘못된 클래스) + FN(놓친 GT)
      - 매칭 안 된 Pred → FP
      - 매칭 안 된 GT  → FN
      - GT-Pred 매칭은 greedy IoU descending
    """
    img_dir = DATA_DIR / "images" / split
    lbl_dir = DATA_DIR / "labels" / split

    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)
    gt_tot = defaultdict(int)
    pd_tot = defaultdict(int)
    confusion = defaultdict(lambda: defaultdict(int))   # gt_cls → pred_cls 잘못 매칭

    for img_path in sorted(img_dir.glob("*.[jp][pn]g")):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]

        # GT bbox 로드 (xyxy 픽셀)
        gts = []
        lbl = lbl_dir / (img_path.stem + ".txt")
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                p = line.split()
                if len(p) != 5:
                    continue
                cid = int(p[0])
                cx, cy, bw, bh = (float(x) for x in p[1:])
                x1 = (cx - bw/2) * W; y1 = (cy - bh/2) * H
                x2 = (cx + bw/2) * W; y2 = (cy + bh/2) * H
                name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
                gts.append({"cls": name, "box": (x1, y1, x2, y2), "matched": False})
                gt_tot[name] += 1

        # Pred bbox
        res = model(img, conf=conf, verbose=False)[0]
        preds = []
        if res.boxes is not None:
            for b in res.boxes:
                cid = int(b.cls[0])
                name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
                xyxy = tuple(b.xyxy[0].cpu().numpy().tolist())
                preds.append({"cls": name, "box": xyxy, "matched": False})
                pd_tot[name] += 1

        # IoU 매트릭스 → greedy descending 매칭
        pairs = []
        for gi, g in enumerate(gts):
            for pi, p in enumerate(preds):
                iou = _iou_xyxy(g["box"], p["box"])
                if iou >= iou_thr:
                    pairs.append((iou, gi, pi))
        pairs.sort(key=lambda x: -x[0])
        for iou, gi, pi in pairs:
            g = gts[gi]; p = preds[pi]
            if g["matched"] or p["matched"]:
                continue
            g["matched"] = True; p["matched"] = True
            if g["cls"] == p["cls"]:
                tp[g["cls"]] += 1
            else:
                fp[p["cls"]]   += 1     # 잘못된 클래스
                fn[g["cls"]]   += 1     # GT 놓침
                confusion[g["cls"]][p["cls"]] += 1

        # 매칭 안 된 것들
        for g in gts:
            if not g["matched"]:
                fn[g["cls"]] += 1
        for p in preds:
            if not p["matched"]:
                fp[p["cls"]] += 1

    print(f"[오류 분석] conf≥{conf}  IoU≥{iou_thr}  split={split}")
    print(f"  {'클래스':<18} {'GT':>5} {'Pred':>5} {'TP':>5} {'FP':>5} {'FN':>5}"
          f"  {'Prec':>6} {'Recall':>7}")
    print("  " + "-" * 65)
    for cls in CLASS_NAMES:
        g = gt_tot.get(cls, 0); pd_ = pd_tot.get(cls, 0)
        t = tp.get(cls, 0);     f_ = fp.get(cls, 0); n = fn.get(cls, 0)
        prec = t / (t + f_) if (t + f_) else 0.0
        rec  = t / (t + n)  if (t + n)  else 0.0
        flag = "  ⚠" if f_ + n > 0 else ""
        print(f"  {cls:<18} {g:>5} {pd_:>5} {t:>5} {f_:>5} {n:>5}"
              f"  {prec:>6.3f} {rec:>7.3f}{flag}")

    # 클래스 혼동 (잘못 매칭된 케이스)
    if confusion:
        print(f"\n  [클래스 혼동] (IoU≥{iou_thr} 인데 클래스가 틀림)")
        for gt_cls, mapping in confusion.items():
            for pred_cls, cnt in mapping.items():
                print(f"    GT={gt_cls:<15} → Pred={pred_cls:<15}  {cnt}회")


analyze_errors(eval_model, split="test", conf=0.25, iou_thr=0.5)


## 11. 추론 속도

In [ ]:
# §6에서 이미 측정했지만, 별도로 다시 확인하고 싶을 때 실행
dummy = np.zeros((640, 640, 3), dtype=np.uint8)
for _ in range(5): eval_model(dummy, verbose=False)
times = []
for _ in range(30):
    t0 = time.perf_counter()
    eval_model(dummy, verbose=False)
    times.append(time.perf_counter() - t0)

avg_ms = sum(times) / len(times) * 1000
fps_re = 1000 / avg_ms
device = "GPU" if torch.cuda.is_available() else "CPU"
print(f"[{device}] 평균={avg_ms:.1f}ms  FPS={fps_re:.1f}  "
      f"min={min(times)*1000:.1f}ms  max={max(times)*1000:.1f}ms")
if fps_re >= 10:
    print("  → 로봇 비전 파이프라인 사용 가능 (10 FPS 기준 충족)")
else:
    print("  ⚠ 10 FPS 미달 — yolov8n 또는 GPU 업그레이드 고려")

## 12. 버전 이력 비교표

In [ ]:
registry2 = ModelRegistry(REGISTRY, VERS_DIR)  # 최신 상태 재로드

print("=" * 105)
print("  전체 버전 이력 (fitness 내림차순)")
print("=" * 105)
registry2.print_history()
print()
print(f"  현재 best: {registry2._data['best_version']}  "
      f"fitness={registry2._data.get('best_fitness', 0)}")
print(f"  레지스트리: {REGISTRY}")

# ── 버전별 fitness 추이 그래프 (2개 이상일 때) ───────────────────────────
vs = registry2._data["versions"]
if len(vs) >= 2:
    def _fit(v):
        m = v["metrics"]
        return m.get("fitness", 0.1*m.get("map50",0) + 0.9*m.get("map5095",0))

    x_labels = [v["version"] for v in vs]
    y_fit    = [_fit(v) for v in vs]
    y_map50  = [v["metrics"].get("map50", 0)   for v in vs]
    y_map95  = [v["metrics"].get("map5095", 0) for v in vs]

    fig, ax = plt.subplots(figsize=(max(6, len(vs)*1.5), 4))
    ax.plot(x_labels, y_fit,   "o-", color="#c0392b", lw=2.5, label="Fitness (best 기준)")
    ax.plot(x_labels, y_map50, "s--", color="#2980b9", lw=1.5, label="mAP@50")
    ax.plot(x_labels, y_map95, "^--", color="#27ae60", lw=1.5, label="mAP@50-95")

    best_idx = y_fit.index(max(y_fit))
    ax.annotate(f"BEST\n{y_fit[best_idx]:.4f}",
                xy=(x_labels[best_idx], y_fit[best_idx]),
                xytext=(0, 14), textcoords="offset points",
                ha="center", fontsize=9, color="#c0392b",
                arrowprops=dict(arrowstyle="->", color="#c0392b"))

    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.set_title("버전별 fitness / mAP 추이")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(BASE_DIR / "version_history.png", dpi=120, bbox_inches="tight")
    plt.show()


---

## 부록: 클래스 추가 시 재학습 정책

**YOLOv8 은 순수 증분학습을 지원하지 않는다** (head 채널 수가 클래스 수에 종속 + catastrophic forgetting). 새 클래스 추가 시 전체 재학습 필요.

### 절차
| 단계 | 내용 |
|------|------|
| 1 | 새 클래스 이미지 수집 + 어노테이션 (기존 클래스도 함께 라벨링된 이미지 권장) |
| 2 | `CLASS_NAMES` 리스트 **끝에** 새 클래스 추가 (기존 id 불변) |
| 3 | 기존 라벨 파일들의 클래스 id 변경 불필요 |
| 4 | `TRAIN_MODE = "more_data"` 로 설정 → 이전 best PT 자동 warm-start (COCO 보다 빠른 수렴) |
| 5 | §5~§6 재실행 → 새 버전 자동 부여 + best 비교 |

```python
TRAIN_MODE        = "more_data"   # 이전 best.pt 에서 warm-start
MORE_DATA_BASE_PT = None          # None = registry best 자동
# CLASS_NAMES.append("new_class")  # cell-2 에서 추가
```

> **주의**: head 채널이 다르면 ultralytics 가 자동으로 새 head 를 생성하고 backbone weight 만 가져온다. 즉 head 는 처음부터 학습된다 — 이 때문에 처음 몇 epoch 의 mAP 가 떨어지는 것은 정상.

### 데이터 권장량
| 클래스 수 | 클래스당 최소 이미지 | 총 권장 train | 권장 epochs |
|-----------|---------------------|---------------|-------------|
| 5 (현재)  | 100장 | 500장 | 100 |
| 8         | 100장 | 800장 | 120 |
| 15+       | 150장 | 2,250장+ | 150 |

### 학습 모드 선택 가이드
| 상황 | TRAIN_MODE | 비고 |
|------|-----------|----|
| 최초 학습 | `"new"` | `BASE_MODEL = "yolov8s.pt"` |
| 학습 중단 후 재개 | `"resume"` | `RESUME_RUN_NAME` 필수 — 같은 데이터, 같은 버전 번호 유지 |
| 데이터 추가됨 | `"more_data"` | 이전 best.pt warm-start, 새 버전 부여 |
| 클래스 추가됨 | `"more_data"` | head 자동 재생성 + backbone warm-start |
| 하이퍼파라미터 실험 | `"new"` 또는 `"more_data"` | 같은 데이터로 ablation 시 `"new"` 권장 |
